# 02 — H1 Results: Agenda Distortion Metrics

Computes all H1 metrics for each outlet vs Tagesschau (mainstream reference).

**Metrics** (all size-controlled or rank-based):

| Metric | What | Literature |
|--------|------|------------|
| Normalized entropy | Topic concentration | Boydstun et al. (2014) |
| JSD | Agenda divergence | DiMaggio et al. (2013) |
| Spearman ρ | Topic priority agreement | McCombs & Shaw (1972) |
| Top-K overlap | Shared top topics | Heidenreich et al. (2019) |
| Coverage breadth (rel.) | Topic range vs expected | Size-controlled binomial model |

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("No .git found")

PROJECT_ROOT = find_project_root(Path.cwd())
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / "agenda_distortion"
OUTPUT_DIR = EXPERIMENT_DIR / "outputs"

for p in [str(EXPERIMENT_DIR), str(PROJECT_ROOT / "1a_BERTopic")]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Loading from: {OUTPUT_DIR}")

## Load Iteration Data

In [ ]:
from modeling import load_iteration

ITERATION_ID = "v1"  # must match notebook 01

result = load_iteration(OUTPUT_DIR, ITERATION_ID)
merged_articles = result.merged_articles
merged_topic_info = result.merged_topic_info

print(f"Loaded iteration '{ITERATION_ID}': {len(merged_articles):,} articles, {result.n_topics} topics")

## H1 Metrics — Per Outlet

In [ ]:
from metrics import compute_all_h1_metrics, format_metrics_table, classify_distortion

metrics_df = compute_all_h1_metrics(merged_articles, top_k=10, min_articles=10)

print("H1 Metrics — All Outlets vs Tagesschau")
print("=" * 80)
display(format_metrics_table(metrics_df))

# Save for thesis
metrics_df.to_csv(OUTPUT_DIR / ITERATION_ID / "h1_metrics.csv", index=False)

## Distortion Classification

- **TYPE A** — breadth restriction (fewer topics than expected)
- **TYPE B** — concentration (high JSD or low entropy)
- **TYPE AB** — both signals present

In [ ]:
classified = classify_distortion(metrics_df)

print("Distortion Type Classification")
print("=" * 50)
for _, row in classified.iterrows():
    print(f"  {row['outlet_label']:<22} → {row['distortion_type']}")

classified.to_csv(OUTPUT_DIR / ITERATION_ID / "h1_classification.csv", index=False)

## Visualizations

In [ ]:
from visualization import plot_h1_metrics_panel, plot_coverage_breadth

fig_dir = OUTPUT_DIR / ITERATION_ID / "figures"

fig = plot_h1_metrics_panel(metrics_df, save_path=fig_dir / "h1_metrics_panel.pdf")
plt.show()

In [ ]:
fig = plot_coverage_breadth(metrics_df, save_path=fig_dir / "h1_coverage_breadth.pdf")
plt.show()

In [ ]:
from visualization import plot_topic_heatmap

fig = plot_topic_heatmap(
    merged_articles, metrics_df,
    normalize="outlet",
    save_path=fig_dir / "h1_heatmap_outlet.pdf",
)
plt.show()

In [ ]:
fig = plot_topic_heatmap(
    merged_articles, metrics_df,
    normalize="topic",
    save_path=fig_dir / "h1_heatmap_topic.pdf",
)
plt.show()

In [ ]:
from visualization import plot_distortion_radar

fig = plot_distortion_radar(metrics_df, save_path=fig_dir / "h1_radar.pdf")
plt.show()

## Bootstrap Confidence Intervals

95% CIs via stratified resampling within each outlet.
CIs that exclude 0 (for JSD) or exclude 1 (for Spearman ρ) indicate statistical reliability.

In [ ]:
from metrics import bootstrap_h1_metrics

print("Running bootstrap (n=1000)... this may take a few minutes.")
bootstrap_df = bootstrap_h1_metrics(merged_articles, n_bootstrap=1000)

# Display JSD CIs
print("\nJSD vs Tagesschau — Bootstrap 95% CIs")
print("=" * 70)
jsd_cis = bootstrap_df[bootstrap_df["metric"] == "jsd_vs_tagesschau"].copy()
jsd_cis = jsd_cis[jsd_cis["outlet_label"] != "Tagesschau"]
display(jsd_cis[["outlet_label", "point_estimate", "ci_lower", "ci_upper"]])

# Display Spearman CIs
print("\nSpearman ρ — Bootstrap 95% CIs")
print("=" * 70)
rho_cis = bootstrap_df[bootstrap_df["metric"] == "spearman_rho"].copy()
rho_cis = rho_cis[rho_cis["outlet_label"] != "Tagesschau"]
display(rho_cis[["outlet_label", "point_estimate", "ci_lower", "ci_upper"]])

bootstrap_df.to_csv(OUTPUT_DIR / ITERATION_ID / "h1_bootstrap.csv", index=False)

In [ ]:
from visualization import plot_bootstrap_cis

for metric in ["jsd_vs_tagesschau", "spearman_rho", "topk_overlap"]:
    fig = plot_bootstrap_cis(
        bootstrap_df, metric,
        save_path=fig_dir / f"h1_bootstrap_{metric}.pdf",
    )
    plt.show()
    plt.close(fig)

## Interpretation

Fill in after running:

**Key findings:**
- Which outlets show strongest agenda distortion?
- Is distortion TYPE A (breadth), TYPE B (concentration), or both?
- Are bootstrap CIs tight enough to be confident?
- Do Spearman ρ values align with JSD (they should be inversely related)?

**Proceed to notebook 03 for robustness checks before documenting in findings.md.**